# CSC-44112 Assessment Part 2
# Customer Response Prediction Using Machine Learning

This notebook is written in simple sections so it can be matched clearly to the report.
Run the notebooks in order from 01 to 04.

## Notebook 4: Results and Evaluation
This notebook evaluates the trained models and creates output graphs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)

PROCESSED_DIR = Path('../data/processed')
MODEL_DIR = Path('../models')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(exist_ok=True)

X_test = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
X_test_scaled = pd.read_csv(PROCESSED_DIR / 'X_test_scaled.csv')
y_test = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze()

### Load trained models

In [ ]:
models = {
    'Logistic Regression': (joblib.load(MODEL_DIR / 'logistic_regression.pkl'), X_test_scaled),
    'Random Forest': (joblib.load(MODEL_DIR / 'random_forest.pkl'), X_test)
}

xgb_path = MODEL_DIR / 'xgboost.pkl'
if xgb_path.exists():
    models['XGBoost'] = (joblib.load(xgb_path), X_test)

models.keys()

### Evaluation metrics table

In [ ]:
results = []

for name, (model, X_eval) in models.items():
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False)
results_df.to_csv('../figures/model_comparison_table.csv', index=False)
results_df

### [Confusion Matrix]
This shows correct and incorrect predictions for each model.

In [ ]:
for name, (model, X_eval) in models.items():
    y_pred = model.predict(X_eval)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.title(f'Confusion Matrix - {name}')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'confusion_matrix_{name.replace(" ", "_").lower()}.png', dpi=300)
    plt.show()

### [ROC Curve]
This compares how well each model separates responders and non-responders.

In [ ]:
for name, (model, X_eval) in models.items():
    y_prob = model.predict_proba(X_eval)[:, 1]
    RocCurveDisplay.from_predictions(y_test, y_prob)
    plt.title(f'ROC Curve - {name}')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'roc_curve_{name.replace(" ", "_").lower()}.png', dpi=300)
    plt.show()

### Classification reports

In [ ]:
for name, (model, X_eval) in models.items():
    print('\n' + '='*60)
    print(name)
    print('='*60)
    y_pred = model.predict(X_eval)
    print(classification_report(y_test, y_pred, zero_division=0))

### [Feature Importance Chart]
Random Forest is used to identify important predictors.

In [ ]:
rf_model = models['Random Forest'][0]
importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance_df.to_csv(FIG_DIR / 'feature_importance_table.csv', index=False)

plt.figure(figsize=(8,5))
plt.barh(importance_df.head(10)['Feature'][::-1], importance_df.head(10)['Importance'][::-1])
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_importance_chart.png', dpi=300)
plt.show()

importance_df.head(10)

### [Learning Curves]
This optional graph helps discuss overfitting and generalisation.

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np

rf_model = models['Random Forest'][0]
train_sizes, train_scores, test_scores = learning_curve(
    rf_model, X_test, y_test, cv=3, scoring='f1', train_sizes=np.linspace(0.2, 1.0, 5)
)

plt.figure(figsize=(6,4))
plt.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Training score')
plt.plot(train_sizes, test_scores.mean(axis=1), marker='o', label='Validation score')
plt.title('Learning Curve - Random Forest')
plt.xlabel('Training Set Size')
plt.ylabel('F1 Score')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'learning_curve_random_forest.png', dpi=300)
plt.show()